<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
public delegate void ProfileUpdateHandler(string message);

public abstract class Customer
{
    protected string _email;

    public int CustomerId { get; protected set; }
    public string Name { get; protected set; }
    public string Email
    {
        get => _email;
        set
        {
            if (string.IsNullOrWhiteSpace(value) || !value.Contains("@"))
                throw new ArgumentException("Некорректный email.");
            _email = value;
        }
    }

    public event ProfileUpdateHandler ProfileUpdated;

    protected Customer(int customerId, string name, string email)
    {
        if (customerId <= 0) throw new ArgumentException("ID должен быть положительным.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Имя не может быть пустым.");
        CustomerId = customerId;
        Name = name;
        Email = email;
    }

    public virtual string GetFullName() => Name;

    public virtual void UpdateEmail(string newEmail)
    {
        Email = newEmail;
        OnProfileUpdated($"Клиент {Name} обновил email на {Email}");
    }

    protected virtual void OnProfileUpdated(string message)
    {
        ProfileUpdated?.Invoke(message);
    }

    public virtual void ViewProfile()
    {
        Console.WriteLine($"ID: {CustomerId} | Имя: {GetFullName()} | Email: {Email}");
    }
}

public class VipCustomer : Customer
{
    public int LoyaltyPoints { get; private set; }
    public string Tier { get; private set; }
    public DateTime VipSince { get; private set; }
    public bool HasDedicatedManager { get; private set; }

    public VipCustomer(int customerId, string name, string email, int loyaltyPoints)
        : base(customerId, name, email)
    {
        if (loyaltyPoints < 0) throw new ArgumentException("Баллы не могут быть отрицательными.");
        LoyaltyPoints = loyaltyPoints;
        VipSince = DateTime.Now.AddYears(-1);
        Tier = loyaltyPoints >= 1000 ? "Platinum" : "Gold";
        HasDedicatedManager = loyaltyPoints >= 500;
    }

    public void AddLoyaltyPoints(int points)
    {
        LoyaltyPoints += points;
        OnProfileUpdated($"VIP {Name} получил {points} баллов. Всего: {LoyaltyPoints}");
    }

    public override void ViewProfile()
    {
        base.ViewProfile();
        Console.WriteLine($"Баллы лояльности: {LoyaltyPoints} | Уровень: {Tier} | Менеджер: {(HasDedicatedManager ? "Да" : "Нет")}");
    }
}

public class RegularCustomer : Customer
{
    public DateTime RegistrationDate { get; protected set; }
    public DateTime? LastEmailUpdate { get; private set; }
    public int PurchaseCount { get; set; }
    public string FavoriteCategory { get; set; } = "Общее";

    public RegularCustomer(int customerId, string name, string email, DateTime registrationDate)
        : base(customerId, name, email)
    {
        RegistrationDate = registrationDate;
    }

    public override void UpdateEmail(string newEmail)
    {
        base.UpdateEmail(newEmail);
        LastEmailUpdate = DateTime.Now;
        OnProfileUpdated($"Обычный клиент {Name} обновил email. Дата: {LastEmailUpdate:yyyy-MM-dd HH:mm}");
    }

    public override void ViewProfile()
    {
        base.ViewProfile();
        Console.WriteLine($"Регистрация: {RegistrationDate:yyyy-MM-dd} | Покупок: {PurchaseCount} | Категория: {FavoriteCategory}");
        if (LastEmailUpdate.HasValue)
            Console.WriteLine($"Последнее обновление email: {LastEmailUpdate:yyyy-MM-dd HH:mm}");
    }
}

public class GroupCustomer : Customer
{
    public string GroupName { get; protected set; }
    public DateTime CreationDate { get; private set; }
    public int MaxMembers { get; private set; } = 50;
    private List<Customer> _members = new List<Customer>();

    public IReadOnlyList<Customer> Members => _members.AsReadOnly();

    public GroupCustomer(int customerId, string groupName, string email)
        : base(customerId, groupName, email)
    {
        GroupName = groupName;
        CreationDate = DateTime.Now;
    }

    public override string GetFullName() => $"Группа «{GroupName}»";

    public void AddMember(Customer member)
    {
        if (member == null) throw new ArgumentNullException(nameof(member));
        if (_members.Count >= MaxMembers) return;
        if (!_members.Contains(member))
        {
            _members.Add(member);
            OnProfileUpdated($"В группу «{GroupName}» добавлен {member.Name}");
        }
    }

    public override void ViewProfile()
    {
        Console.WriteLine($"Группа ID: {CustomerId} | Название: {GroupName} | Email: {Email}");
        Console.WriteLine($"Создана: {CreationDate:yyyy-MM-dd} | Участников: {_members.Count}/{MaxMembers}");
        if (_members.Any())
        {
            Console.WriteLine("Участники:");
            foreach (var m in _members)
                Console.WriteLine($"  → {m.Name} (ID: {m.CustomerId})");
        }
    }
}

void OnProfileUpdated(string message)
{
    Console.WriteLine($"Уведомление: {message}");
}

var vip = new VipCustomer(1, "Алексей", "alex@vip.com", 800);
var regular = new RegularCustomer(2, "Мария", "maria@test.com", new DateTime(2023, 5, 10));
var group = new GroupCustomer(100, "Команда Alpha", "alpha@org.com");

vip.ProfileUpdated += OnProfileUpdated;
regular.ProfileUpdated += OnProfileUpdated;
group.ProfileUpdated += OnProfileUpdated;

List<Customer> customerList = new List<Customer> { vip, regular, group };
Dictionary<int, Customer> customerDict = new Dictionary<int, Customer>
{
    { vip.CustomerId, vip },
    { regular.CustomerId, regular },
    { group.CustomerId, group }
};

foreach (var customer in customerList)
{
    customer.ViewProfile();
    Console.WriteLine();
}

regular.UpdateEmail("maria.new@test.com");

group.AddMember(vip);
group.AddMember(regular);

var vips = customerList.OfType<VipCustomer>().Where(c => c.LoyaltyPoints > 500).ToList();
Console.WriteLine($"\nНайдено VIP-клиентов с >500 баллов: {vips.Count}");

ID: 1 | Имя: Алексей | Email: alex@vip.com
Баллы лояльности: 800 | Уровень: Gold | Менеджер: Да

ID: 2 | Имя: Мария | Email: maria@test.com
Регистрация: 2023-05-10 | Покупок: 0 | Категория: Общее

Группа ID: 100 | Название: Команда Alpha | Email: alpha@org.com
Создана: 2025-11-17 | Участников: 0/50

Уведомление: Клиент Мария обновил email на maria.new@test.com
Уведомление: Обычный клиент Мария обновил email. Дата: 2025-11-17 18:54
Уведомление: В группу «Команда Alpha» добавлен Алексей
Уведомление: В группу «Команда Alpha» добавлен Мария

Найдено VIP-клиентов с >500 баллов: 1
